# FabriSense: Fabric Classification for iBUG Dataset
### Transfer learning + CV heuristics for ~2000 samples, 20 fabric classes

**Dataset:** [iBUG Fabrics Dataset](https://www.kaggle.com/datasets/orchit/the-fabrics-dataset-by-ibug)
- ~2000 garment samples
- 20 fabric classes with imbalanced distribution
- 4 illumination images per garment (photometric stereo)

In [ ]:
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from PIL import Image
import colorsys
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
class Config:
    IMG_SIZE = 224
    BATCH_SIZE = 32
    EPOCHS = 25
    LEARNING_RATE = 1e-4
    ARCHITECTURE = "efficientnet_b0"
    ILLUMINATION_COUNT = 4
    OUTPUT_DIR = Path("./models")

    FABRIC_CLASSES = [
        "Acrylic", "Chenille", "Corduroy", "Cotton", "Crepe",
        "Denim", "Felt", "Fleece", "Leather", "Linen",
        "Lut", "Nylon", "Polyester", "Satin", "Silk",
        "Suede", "Terrycloth", "Velvet", "Viscose", "Wool"
    ]

    CLASS_WEIGHTS = {
        "Acrylic": 83, "Chenille": 77, "Corduroy": 42, "Cotton": 1,
        "Crepe": 50, "Denim": 7, "Felt": 250, "Fleece": 30,
        "Leather": 53, "Linen": 53, "Lut": 250, "Nylon": 17,
        "Polyester": 4, "Satin": 42, "Silk": 28, "Suede": 100,
        "Terrycloth": 33, "Velvet": 45, "Viscose": 27, "Wool": 5
    }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Total classes: {len(Config.FABRIC_CLASSES)}")
CLASS_TO_IDX = {c: i for i, c in enumerate(Config.FABRIC_CLASSES)}

In [ ]:
data_dir = Path("/kaggle/input/the-fabrics-dataset-by-ibug")

class IBuGDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        entry = self.samples[idx]
        images = []
        for img_path in entry['images']:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            images.append(img)
        # Average illuminations
        if len(images) > 1:
            images = torch.stack(images).mean(dim=0)
        return images, entry['label']

# Load garment data
train_dir = data_dir / "train"
samples = []
for garment_folder in train_dir.iterdir():
    if not garment_folder.is_dir():
        continue
    images = sorted(garment_folder.glob("*.jpg"))[:Config.ILLUMINATION_COUNT]
    if images:
        samples.append({'images': images, 'label': None, 'gid': garment_folder.name})

print(f"Loaded {len(samples)} garments")

In [ ]:
def build_model(architecture, num_classes):
    if architecture == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif architecture == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported: {architecture}")
    return model

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.3),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
# Train/val split
train_s, val_s = train_test_split(samples, test_size=0.2, random_state=42)
train_loader = DataLoader(IBuGDataset(train_s, train_tf), batch_size=Config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(IBuGDataset(val_s, val_tf), batch_size=Config.BATCH_SIZE)

model = build_model(Config.ARCHITECTURE, len(Config.FABRIC_CLASSES)).to(device)
weights = torch.tensor([Config.CLASS_WEIGHTS.get(c, 1.0) for c in Config.FABRIC_CLASSES], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += out.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def validate(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for images, lbls in loader:
            out = model(images.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.numpy())
    return f1_score(labels, preds, average='macro'), np.mean(np.array(preds) == np.array(labels))

In [ ]:
best_f1 = 0.0
for epoch in range(Config.EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_f1, val_acc = validate(model, val_loader, device)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val F1={val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), Config.OUTPUT_DIR / f"best_{Config.ARCHITECTURE}.pth")
    scheduler.step()
print(f"Done. Best F1: {best_f1:.4f}")

In [ ]:
def compute_metrics(image):
    sample = image.resize((256, 256))
    rgb = np.array(sample).astype(np.float32)
    gray = np.array(sample.convert('L')).astype(np.float32)
    gx, gy = np.abs(np.diff(gray, axis=1)), np.abs(np.diff(gray, axis=0))
    texture = ((gx.mean() + gy.mean()) / 2) / 255 * 100
    rows = np.std(gray.mean(axis=1)) / 255 * 100
    cols = np.std(gray.mean(axis=0)) / 255 * 100
    sheen = []
    for r, g, b in (rgb / 255).reshape(-1, 3)[::64]:
        _, s, v = colorsys.rgb_to_hsv(float(r), float(g), float(b))
        sheen.append(v * (1 - s / 2))
    return {'texture': texture, 'rows': rows, 'cols': cols, 'sheen': np.mean(sheen) * 100}

def infer_pattern(m):
    if max(m['rows'], m['cols']) < 5: return "Solid"
    if m['cols'] > m['rows'] * 1.35: return "Vertical Stripes"
    if m['rows'] > m['cols'] * 1.35: return "Horizontal Stripes"
    if min(m['rows'], m['cols']) > 6: return "Plaid"
    return "Textured"

In [ ]:
model.load_state_dict(torch.load(Config.OUTPUT_DIR / f"best_{Config.ARCHITECTURE}.pth"))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for images, lbls in val_loader:
        out = model(images.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
print(classification_report(labels, preds, target_names=Config.FABRIC_CLASSES))
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")

In [ ]:
def predict(image_path):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        prob, idx = torch.softmax(out, dim=1).max(1)
    return {'fabric': Config.FABRIC_CLASSES[idx.item()], 'confidence': prob.item(), 'pattern': infer_pattern(compute_metrics(img))}